# Train Maturity Model — Step by Step

This notebook trains a deep learning model that classifies fruit maturity into 3 stages: Unripe, Ripe, Overripe. The model works for tomato, mango, and strawberry — YOLO handles crop detection separately.

**How to use this notebook:** Run each cell in order, top to bottom. Do not skip cells. To run a cell, click on it and press Shift+Enter, or click the play button on the left.

**Before you start:**
1. Enable GPU: top-right corner → `Connect` → `Change runtime type` → Hardware accelerator = `T4 GPU` → Save.
2. Download the dataset zip from Kaggle (instructions in Step 1 below).
3. Total time: about 30-40 minutes (mostly waiting for training).

## Step 1 — Download the dataset from Kaggle

Do this in your browser first, before running any cells:

1. Go to: https://www.kaggle.com/datasets/asadullahprl/fruits-ripeness-classification-dataset
2. Sign in with a free Kaggle account (sign up if you don't have one).
3. Click the **Download** button at the top right. A zip file downloads (about 200-400 MB).
4. The file is usually named `archive.zip`. Remember where it saved (usually your Downloads folder).

This dataset contains 5 fruits (Apple, Banana, Mango, Orange, Tomato) with 3 ripeness stages each (Unripe, Ripe, Overripe). The model learns the general ripeness pattern and generalizes to strawberry even though strawberry is not in the training data.

You do NOT need a Kaggle API token. You do NOT need to authenticate anything in Colab. Just download the zip manually and upload it in Step 3.

## Step 2 — Install packages

Run this cell. Wait for it to finish (about 1-2 minutes). You will see text scrolling — that is normal. When you see a number in brackets `[1]` next to the cell, it is done.

In [ ]:
!pip install -q tensorflow scikit-learn matplotlib seaborn Pillow
print('Packages installed.')

## Step 3 — Upload the dataset zip

Run this cell. A button appears that says `Choose Files` or `Browse...`. Click it and select the zip file you downloaded from Kaggle in Step 1.

The upload takes 2-5 minutes depending on your internet speed. Wait until you see the message saying the file was uploaded and extracted.

In [ ]:
from google.colab import files
import os

print('Click Choose Files and select the Kaggle zip file you downloaded.')
print('Upload takes 2-5 minutes. Wait for it to finish.')
print()

uploaded = files.upload()

zip_name = None
for fn in uploaded.keys():
    if fn.lower().endswith('.zip'):
        zip_name = fn
        break

if not zip_name:
    raise ValueError('No zip file found. Please upload the dataset zip from Kaggle.')

print(f'Uploaded: {zip_name} ({len(uploaded[zip_name]) / 1024 / 1024:.1f} MB)')

import zipfile
os.makedirs('dataset', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('dataset/')

print('Dataset extracted to dataset/ folder.')

## Step 4 — Look at the dataset structure

Run this cell. It shows you what folders are in the dataset so we know how to load the images.

In [ ]:
import os

print('Dataset structure:')
print()
for root, dirs, files in os.walk('dataset'):
    level = root.replace('dataset', '').count(os.sep)
    indent = '  ' * level
    folder_name = os.path.basename(root) or 'dataset'
    image_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if image_files:
        print(f'{indent}{folder_name}/  ({len(image_files)} images)')
    elif dirs:
        print(f'{indent}{folder_name}/')
    if level < 3:
        for d in sorted(dirs)[:10]:
            sub_path = os.path.join(root, d)
            sub_images = []
            for r2, d2, f2 in os.walk(sub_path):
                sub_images.extend([f for f in f2 if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            if sub_images:
                print(f'{indent}  {d}/  ({len(sub_images)} images)')

## Step 5 — Load and prepare the data

Run this cell. It loads all images, resizes them to 224x224 (what MobileNetV2 expects), and splits them into training and test sets. Takes 1-3 minutes depending on dataset size.

In [ ]:
import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.utils import to_categorical

IMG_SIZE = 224

# Find all subfolders that contain images — these are our classes
classes = []
for root, dirs, files in os.walk('dataset'):
    image_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if image_files:
        class_name = os.path.basename(root)
        if class_name not in classes:
            classes.append(class_name)

print(f'Found classes: {classes}')
print(f'Number of classes: {len(classes)}')
print()

# Load all images
images = []
labels = []

for root, dirs, files in os.walk('dataset'):
    image_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if not image_files:
        continue
    class_name = os.path.basename(root)
    for img_file in image_files:
        try:
            img_path = os.path.join(root, img_file)
            img = Image.open(img_path).convert('RGB')
            img = img.resize((IMG_SIZE, IMG_SIZE))
            images.append(np.array(img))
            labels.append(class_name)
        except Exception as e:
            pass

images = np.array(images)
labels = np.array(labels)

print(f'Total images loaded: {len(images)}')
print(f'Image shape: {images.shape}')
print()
print('Images per class:')
for c in classes:
    count = (labels == c).sum()
    print(f'  {c}: {count}')

# Encode labels to numbers
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)
labels_categorical = to_categorical(labels_encoded)

# Split into train and test (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    images, labels_categorical, test_size=0.2, random_state=42, stratify=labels_encoded
)

print()
print(f'Training images: {len(X_train)}')
print(f'Test images: {len(X_test)}')
print(f'Class names: {list(label_encoder.classes_)}')

## Step 6 — Build the model

Run this cell. It builds a MobileNetV2 model with transfer learning. MobileNetV2 is a small, fast model that works well on images and runs on a laptop without a powerful GPU.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

num_classes = len(classes)

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()
print()
print(f'Model built. {num_classes} classes: {list(label_encoder.classes_)}')

## Step 7 — Train the model

Run this cell. Training takes 10-20 minutes. You will see progress bars and accuracy numbers updating. **Wait for it to finish completely** before moving to the next cell.

The `epochs=15` means it goes through the data 15 times. More epochs usually means better accuracy but slower.

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    steps_per_epoch=len(X_train) // 32,
    epochs=15,
    validation_data=(X_test, y_test),
    verbose=1
)

print('Training complete!')

## Step 8 — Check the accuracy

Run this cell. It shows the final accuracy on the test set and a plot of how accuracy improved during training. Take a screenshot of the output for your report.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Final test accuracy: {test_acc * 100:.2f}%')
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Training')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Training')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.show()

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print()
print('Classification report:')
print(classification_report(y_true_classes, y_pred_classes, target_names=label_encoder.classes_))

cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Step 9 — Save the model and class labels

Run this cell. It saves two files:
- `maturity_model.h5` — the trained model (about 15 MB)
- `class_labels.json` — the list of class names so the engine knows what each output means

In [ ]:
import json

model.save('maturity_model.h5')
print('Model saved as maturity_model.h5')

with open('class_labels.json', 'w') as f:
    json.dump(list(label_encoder.classes_), f)
print('Class labels saved as class_labels.json')
print(f'Classes: {list(label_encoder.classes_)}')
print()
print('Files in current directory:')
!ls -lh maturity_model.h5 class_labels.json

## Step 10 — Download the files to your laptop

Run this cell. Two download buttons appear — click each one to download `maturity_model.h5` and `class_labels.json` to your laptop.

In [ ]:
from google.colab import files

print('Click the buttons below to download each file:')
print()
print('1. Download the model:')
files.download('maturity_model.h5')
print()
print('2. Download the class labels:')
files.download('class_labels.json')

## Done

You now have two files:
- `maturity_model.h5` — the trained model (about 15 MB)
- `class_labels.json` — the list of classes (small text file)

Put both files in your `maturity_engine/models/` folder. Then replace `engine.py` with the new version from the zip and update `crops.yaml`. Full instructions are in `Step_By_Step_Guide.docx`.